# POS-Auswertung gegen das blind erhobene Wortarten-Gold



## Schritt 0 — Konfiguration


In [ ]:
from pathlib import Path

#ADJA/ADJD oder nur ADJ
FEINE_EBENE = True

GOLD_PFAD = (Path("..") / "04_konsolidiert"
             / ("goldstandard_v1_pos_final.conll" if FEINE_EBENE
                else "goldstandard_v1_pos_final_upos.conll"))
SYSTEME_CONLL  = {"ensemble": Path("..") / "05_ergebnisse" / "pos"
                              / "ensemble_pred.conll"}
SYSTEME_MODELL = {"finetuned": "../../output/model-best",
                  "lg": "de_core_news_lg",
                  "md": "de_core_news_md"}
REPS, SEED = 1000, 42
AUSGABE = Path("..") / "05_ergebnisse" / "pos"
AUSGABE.mkdir(parents=True, exist_ok=True)

UNBEARBEITET = "_"

MIN_CLUSTER = 8

IAA_MITTEL = 0.940         
IAA_BESTE  = 0.974         
IAA_PAARWEISE = 0.907      
IAA_ADJ_FEIN = 0.963       

print("Gold:", GOLD_PFAD, "| Systeme:",
      list(SYSTEME_CONLL) + list(SYSTEME_MODELL))

Gold: ../04_konsolidiert/goldstandard_v1_pos_final.conll | Systeme: ['ensemble', 'finetuned', 'lg', 'md']


## Schritt 1 — Einlesen und Tokengleichheit sichern


In [ ]:
def lese_conll(pfad):
    saetze, cur, meta = [], [], {}
    for nr, roh in enumerate(open(pfad, encoding="utf-8"), 1):
        z = roh.rstrip("\n")
        if z.startswith("#"):
            if "=" in z:
                k, _, v = z.lstrip("#").partition("=")
                meta[k.strip()] = v.strip()
            continue
        if not z.strip():
            if cur:
                saetze.append(dict(erzaehlung=meta.get("erzaehlung", "?"),
                                   sammlung=meta.get("sammlung", "?"),
                                   tokens=cur))
                cur = []
            continue
        p = z.split("\t")
        if len(p) != 3:
            raise ValueError(f"{pfad}:{nr}: 3 Spalten erwartet: {z!r}")
        cur.append((p[0], p[1]))
    if cur:
        saetze.append(dict(erzaehlung=meta.get("erzaehlung", "?"),
                           sammlung=meta.get("sammlung", "?"), tokens=cur))
    return saetze

def pruefe_tokengleich(gold, andere, name):
    g = [t for s in gold for t, _ in s["tokens"]]
    a = [t for s in andere for t, _ in s["tokens"]]
    if g != a:
        raise ValueError(f"Tokenfolge von '{name}' weicht vom Gold ab.")

gold = lese_conll(GOLD_PFAD)
n_tok = sum(len(s["tokens"]) for s in gold)

auswertbar = [[gt != UNBEARBEITET for _, gt in s["tokens"]] for s in gold]
n_aus = sum(sum(m) for m in auswertbar)
erz_aus = {s["erzaehlung"] for s, m in zip(gold, auswertbar) if any(m)}

print(f"Ebene: {'fein (ADJA/ADJD)' if FEINE_EBENE else 'zusammengefasst (ADJ)'}")
print(f"Gold: {len(gold)} Sätze | {n_tok} Tokens | "
      f"{len({s['erzaehlung'] for s in gold})} Erzählungen")
print(f"Davon bearbeitet: {n_aus} Tokens ({n_aus / n_tok:.0%}) "
      f"aus {len(erz_aus)} Erzählungen")
if n_aus < n_tok:
    print()
    print("HINWEIS: Die Wortartenebene wurde nicht vollständig erhoben.")
    print("Sämtliche Kennzahlen beziehen sich allein auf die bearbeiteten")
    print(f"{n_aus} Token; sie sind deshalb nicht mit den Werten der")
    print("Figurenebene vergleichbar, die auf allen 4137 Token beruht.")
assert n_aus, "Kein einziges Token trägt ein Tag."

Ebene: fein (ADJA/ADJD)
Gold: 150 Sätze | 4137 Tokens | 20 Erzählungen
Davon bearbeitet: 1794 Tokens (43%) aus 20 Erzählungen

HINWEIS: Die Wortartenebene wurde nicht vollständig erhoben.
Sämtliche Kennzahlen beziehen sich allein auf die bearbeiteten
1794 Token; sie sind deshalb nicht mit den Werten der
Figurenebene vergleichbar, die auf allen 4137 Token beruht.


## Schritt 2 — Systemvorhersagen beschaffen


In [ ]:
def tag_von(token):
    if token.tag_ in ("ADJA", "ADJD"):
        return token.tag_ if FEINE_EBENE else "ADJ"
    return token.pos_ or token.tag_


def modell_vorhersage(modellpfad):
    import spacy
    from spacy.tokens import Doc
    nlp = spacy.load(modellpfad)
    schwer = [n for n in ("parser", "lemmatizer", "attribute_ruler", "ner")
              if n in nlp.pipe_names]
    saetze = []
    with nlp.select_pipes(disable=schwer):
        for s in gold:
            doc = Doc(nlp.vocab, words=[t for t, _ in s["tokens"]])
            for _, proc in nlp.pipeline:
                doc = proc(doc)
            saetze.append(dict(erzaehlung=s["erzaehlung"],
                               sammlung=s["sammlung"],
                               tokens=[(t.text, tag_von(t)) for t in doc]))
    return saetze

systeme = {}
fehlend = {n: pf for n, pf in SYSTEME_CONLL.items() if not pf.exists()}
for n, pf in fehlend.items():
    print(f"  '{n}' uebersprungen: {pf} nicht gefunden.")
for name, pfad in {n: pf for n, pf in SYSTEME_CONLL.items()
                   if pf.exists()}.items():
    try:
        s = lese_conll(pfad); pruefe_tokengleich(gold, s, name)
        systeme[name] = s; print(f"  Datei-System: {name}")
    except Exception as e:
        print(f"  WARNUNG '{name}': {e} — übersprungen.")
for name, pfad in SYSTEME_MODELL.items():
    try:
        systeme[name] = modell_vorhersage(pfad)
        print(f"  Modell-System: {name}")
    except Exception as e:
        print(f"  WARNUNG '{name}': {type(e).__name__}: {e} — übersprungen.")
#Auf der feinen Ebene sind nur Systeme auswertbar, die ADJA und ADJD
#ueberhaupt ausgeben. Fehlt die Unterscheidung, entstuende ein
#unverdient schlechter Wert, weil jedes Adjektiv als Fehler zaehlte.
if FEINE_EBENE:
    ohne_feine = [n for n, s in systeme.items()
                  if not {t for satz in s for _, t in satz["tokens"]}
                          & {"ADJA", "ADJD"}]
    for n in ohne_feine:
        del systeme[n]
        print(f"  '{n}' von der feinen Auswertung ausgeschlossen: "
              f"gibt ADJA/ADJD nicht aus.")
    if ohne_feine:
        print("  Die Unterscheidung wird von diesen Systemen nicht "
              "geliefert; ein Wert waere nicht aussagekraeftig.")

assert systeme, "kein System verfügbar"

  Datei-System: ensemble
  Modell-System: finetuned
  Modell-System: lg
  Modell-System: md
  'ensemble' von der feinen Auswertung ausgeschlossen: gibt ADJA/ADJD nicht aus.
  'finetuned' von der feinen Auswertung ausgeschlossen: gibt ADJA/ADJD nicht aus.
  Die Unterscheidung wird von diesen Systemen nicht geliefert; ein Wert waere nicht aussagekraeftig.


## Schritt 3 — Metrik-Bausteine


In [ ]:
import math
import numpy as np
from collections import Counter, defaultdict

def zaehle(system):
    z = defaultdict(lambda: np.zeros(2, dtype=np.int64))
    maske, konf = [], Counter()
    adj = dict(tp=0, fp=0, fn=0)
    for g, s, m in zip(gold, system, auswertbar):
        for ((_, gt), (_, st), nutzen) in zip(g["tokens"], s["tokens"], m):
            if not nutzen:
                continue
            ok = gt == st
            z[g["erzaehlung"]] += (1 if ok else 0, 1)
            maske.append(ok)
            if not ok: konf[(gt, st)] += 1
            #Auf der feinen Ebene zaehlen ADJA und ADJD gemeinsam als
            #Adjektiv, damit die Kennzahl zwischen beiden Ebenen
            #vergleichbar bleibt.
            ist_adj = lambda x: x in ("ADJ", "ADJA", "ADJD")
            if ist_adj(gt) and ist_adj(st): adj["tp"] += 1
            elif ist_adj(st): adj["fp"] += 1
            elif ist_adj(gt): adj["fn"] += 1
    return z, np.array(maske, dtype=bool), konf, adj

def cluster_ci(z, rng):
    #Erzaehlungen ohne bearbeitete Token gehen nicht ein
    erz = [e for e in sorted(z) if z[e][1] > 0]
    if len(erz) < 2:
        return float("nan"), float("nan")
    arr = np.stack([z[e] for e in erz])
    idx = rng.integers(0, len(arr), size=(REPS, len(arr)))
    s = arr[idx].sum(axis=1)
    lo, hi = np.percentile(s[:, 0] / s[:, 1], [2.5, 97.5])
    return float(lo), float(hi)

def delta_ci(za, zb, rng):
    erz = [e for e in sorted(set(za) | set(zb))
           if za.get(e, np.zeros(2))[1] > 0 and zb.get(e, np.zeros(2))[1] > 0]
    if len(erz) < 2:
        return float("nan"), float("nan")
    a = np.stack([za[e] for e in erz]); b = np.stack([zb[e] for e in erz])
    idx = rng.integers(0, len(erz), size=(REPS, len(erz)))
    sa, sb = a[idx].sum(axis=1), b[idx].sum(axis=1)
    lo, hi = np.percentile(sa[:, 0]/sa[:, 1] - sb[:, 0]/sb[:, 1], [2.5, 97.5])
    return float(lo), float(hi)

def mcnemar(ma, mb):
    b = int(np.sum(ma & ~mb)); c = int(np.sum(~ma & mb)); n = b + c
    if n == 0: return b, c, 1.0
    if n <= 100:
        k = min(b, c)
        p = min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2**n)
    else:
        z = (abs(b - c) - 1) / math.sqrt(n)
        p = math.erfc(z / math.sqrt(2))
    return b, c, p

print("Metrik-Bausteine definiert.")

Metrik-Bausteine definiert.


## Schritt 4 — Systemberichte


In [ ]:
rng = np.random.default_rng(SEED)
Z, M, bericht = {}, {}, {"n_tokens": n_tok, "systeme": {}, "vergleiche": []}

for name, s in systeme.items():
    z, maske, konf, adj = zaehle(s)
    Z[name], M[name] = z, maske
    acc = float(maske.mean()); lo, hi = cluster_ci(z, rng)
    p = adj["tp"]/(adj["tp"]+adj["fp"]) if adj["tp"]+adj["fp"] else 0
    r = adj["tp"]/(adj["tp"]+adj["fn"]) if adj["tp"]+adj["fn"] else 0
    f = 2*p*r/(p+r) if p+r else 0
    js = defaultdict(lambda: np.zeros(2, dtype=np.int64))
    je_erz = defaultdict(int)
    for g, sy, m in zip(gold, s, auswertbar):
        for ((_, gt), (_, st), nutzen) in zip(g["tokens"], sy["tokens"], m):
            if not nutzen:
                continue
            js[g["sammlung"]] += (1 if gt == st else 0, 1)
            je_erz[(g["sammlung"], g["erzaehlung"])] += 1
    ci_txt = (f"[{lo:.4f}, {hi:.4f}]" if lo == lo else "— (zu wenige Cluster)")
    print(f"\n[{name}] Accuracy {acc:.4f}  95%-CI {ci_txt}  "
          f"(n = {int(maske.size)} bearbeitete Token)")
    for samm, v in sorted(js.items()):
        print(f"    {samm:<10} {v[0]/v[1]:.4f} (n={v[1]})")
    print(f"    ADJ: P={p:.3f} R={r:.3f} F1={f:.3f}")
    fehler_gesamt = sum(konf.values())
    print(f"    Verwechslungen ({fehler_gesamt} Fehler bei "
          f"{int(maske.size)} Token):")
    kumuliert = 0
    for (a, bb), n in konf.most_common():
        kumuliert += n
        print(f"      {a:>6} -> {bb:<6} {n:>4}  "
              f"{n / fehler_gesamt:>5.1%}  kumuliert {kumuliert / fehler_gesamt:>5.1%}")
    bericht["systeme"].setdefault(name, {})

    knapp = sorted({samm for (samm, _) in je_erz}
                   - {samm for samm in {s2 for (s2, _) in je_erz}
                      if sum(1 for (a, _) in je_erz if a == samm) >= MIN_CLUSTER})
    if knapp:
        print(f"    HINWEIS: weniger als {MIN_CLUSTER} Erzählungen bei "
              + ", ".join(knapp) + "; Sammlungswerte nur als Richtungsangabe.")
    bericht["systeme"][name] = dict(
        accuracy=acc, ci95=[lo, hi], n_bewertet=int(maske.size),
        adj=dict(precision=p, recall=r, f1=f),
        konfusion=[[list(k), n] for k, n in konf.most_common()],
        n_fehler=int(sum(konf.values())))

    if IAA_MITTEL is not None:
        lage = ("unter dem Mittel" if acc <= IAA_MITTEL
                else "über dem Mittel" if acc <= IAA_BESTE
                else "über der besten Bearbeitung")
        print(f"    Agreement: {acc:.4f} liegt {lage} "
              f"(Mittel {IAA_MITTEL:.3f}, beste {IAA_BESTE:.3f}).")
        if acc > IAA_BESTE:
            print("    WARNUNG: Der Wert übertrifft die beste menschliche")
            print("    Bearbeitung. Das belegt keine Überlegenheit, sondern")
            print("    deutet auf einen unscharfen Maßstab oder auf")
            print("    Trainingsmaterial im Goldstandard hin.")
    if IAA_ADJ_FEIN is not None:
        print(f"    (Agreement der ADJA/ADJD-Unterscheidung: "
              f"{IAA_ADJ_FEIN:.3f} bei n = 90; gilt fuer die feine Fassung "
              f"und damit fuer die Spalte 'konstruktion'.)")


[lg] Accuracy 0.9326  95%-CI [0.9129, 0.9485]  (n = 1794 bearbeitete Token)
    Andersen   0.9326 (n=371)
    Arabisch   0.9074 (n=443)
    Grimm      0.9206 (n=403)
    Japanisch  0.9601 (n=577)
    ADJ: P=0.874 R=0.918 F1=0.896
    Verwechslungen (121 Fehler bei 1794 Token):
        PART -> ADV      16  13.2%  kumuliert 13.2%
        PART -> ADP      13  10.7%  kumuliert 24.0%
         ADV -> SCONJ     8   6.6%  kumuliert 30.6%
        VERB -> NOUN      7   5.8%  kumuliert 36.4%
         AUX -> VERB      6   5.0%  kumuliert 41.3%
        NOUN -> ADJA      6   5.0%  kumuliert 46.3%
        NOUN -> PROPN     6   5.0%  kumuliert 51.2%
       CCONJ -> ADV       5   4.1%  kumuliert 55.4%
        INTJ -> PROPN     5   4.1%  kumuliert 59.5%
         DET -> PRON      5   4.1%  kumuliert 63.6%
       SCONJ -> ADP       3   2.5%  kumuliert 66.1%
       PROPN -> NOUN      3   2.5%  kumuliert 68.6%
        PRON -> DET       2   1.7%  kumuliert 70.2%
        ADJD -> NOUN      2   1.7%  kumuliert

## Schritt 5 — Gepaarte Vergleiche


In [6]:
import json
namen = list(systeme)
for i in range(len(namen)):
    for j in range(i + 1, len(namen)):
        a, b = namen[i], namen[j]
        d = float(M[a].mean() - M[b].mean())
        lo, hi = delta_ci(Z[a], Z[b], rng)
        nb_, nc, pw = mcnemar(M[a], M[b])
        sig = "  *" if not (lo <= 0 <= hi) else ""
        print(f"Δ {a} − {b}: {d:+.4f}  CI [{lo:+.4f}, {hi:+.4f}]{sig}  "
              f"| McNemar b={nb_}, c={nc}, p={pw:.4f}")
        bericht["vergleiche"].append(dict(a=a, b=b, delta=d,
                                          ci95=[lo, hi],
                                          mcnemar=dict(b=nb_, c=nc, p=pw)))
(AUSGABE / "pos_eval.json").write_text(
    json.dumps(bericht, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\nGespeichert: {AUSGABE / 'pos_eval.json'}")

Δ lg − md: +0.0006  CI [-0.0087, +0.0091]  | McNemar b=25, c=24, p=1.0000

Gespeichert: ../05_ergebnisse/pos/pos_eval.json
